# MICS DDML analysis — Major-overlap robustness

This notebook prepares the household and child samples, estimates the DoubleML models, and builds the reported tables and diagnostics. Fitted models and result files are saved so completed work does not need to be repeated.


## Setup

### Project paths


In [ ]:
from pathlib import Path
import os
import pickle
from warnings import filterwarnings


ROOT = Path("../../").resolve()
DATA = ROOT / "Data" / "3. Final"
OUT = ROOT / "Output"
FIGS = ROOT / "Figures"
MODELS = OUT / "models" / "major_overlap"
TABLES = ROOT / "Writing edit" / "Table"

for folder in [OUT, FIGS, MODELS, TABLES]:
    folder.mkdir(parents=True, exist_ok=True)

HH_FILE = DATA / "MASTER_MICS_FINAL.dta"
U5_FILE = DATA / "MASTER_MICS_FINAL_U5.dta"

if not HH_FILE.is_file() or not U5_FILE.is_file():
    raise FileNotFoundError("The HH or U5 source file is missing.")

filterwarnings("ignore")

# Limit native libraries before importing NumPy and the estimators.
for variable in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ[variable] = "1"

os.environ["MPLCONFIGDIR"] = "/tmp/mics_ddml_matplotlib"
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

print(ROOT)


### Imports


In [ ]:
import doubleml as dml
import matplotlib.pyplot as plt
import narwhals._interchange
import numpy as np
import pandas as pd
import pyreadstat
import statsmodels.api as sm
import sklearn

from IPython.display import Markdown, display
from joblib import hash as joblib_hash
from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    StackingClassifier,
    StackingRegressor,
)
from sklearn.linear_model import (
    ElasticNetCV,
    LassoCV,
    LinearRegression,
    LogisticRegression,
    LogisticRegressionCV,
    RidgeCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from xgboost import XGBClassifier, XGBRegressor


### Runtime and parallelism


In [ ]:
SEED = 42
FOLDS = 5
IRM_REPS = 3
APOS_REPS = 1
TRIM = 0.01

LEVELS = {
    0: "No Treatment",
    1: "Boil",
    2: "Chlorine",
    3: "Straining/settling",
}

CPU = os.cpu_count() or 1
WORKERS = max(1, CPU - 1)
IRM_WORKERS = min(WORKERS, FOLDS)
APOS_WORKERS = min(WORKERS, len(LEVELS))
TUNING_WORKERS = max(1, WORKERS // max(IRM_WORKERS, APOS_WORKERS))

LEARNER_NAMES = [
    "ols",
    "lasso",
    "ridge",
    "enet",
    "rf",
    "xgb",
    "stacked",
]

display(
    pd.DataFrame(
        {
            "available_cpus": [CPU],
            "irm_fold_workers": [IRM_WORKERS],
            "apos_level_workers": [APOS_WORKERS],
            "inner_tuning_workers": [TUNING_WORKERS],
            "direct_model_workers": [WORKERS],
        }
    )
)


## Data

### Household variables


In [ ]:
def base(raw):
    '''Create variables shared by the HH and U5 samples.'''

    needed = [
        "windex5",
        "urban",
        "WS1_g",
        "country_cat",
        "wq27_decile",
        "Any_U5",
        "Girls_less_than15",
        "Boys_15or_less",
        "RiskSource",
        "RiskHome",
        "SomeRiskHome",
        "VeryHighRiskHome",
        "WQ15_g",
        "Toilet",
        "HHCHILDREN",
    ]
    missing = sorted(set(needed) - set(raw.columns))

    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    df = raw.copy()

    for q in range(1, 6):
        df[f"wealth_q{q}"] = (df["windex5"] == q).astype("int8")

    df["urban_bin"] = df["urban"].astype("int8")
    df["num_children"] = df["HHCHILDREN"].fillna(0).clip(upper=10).astype(int)
    df["num_children_bin"] = df["num_children"].clip(upper=4).astype(int)

    ws = pd.get_dummies(df["WS1_g"].astype("Int64"), prefix="ws1g", dtype=int)
    country = pd.get_dummies(
        df["country_cat"].astype("Int64"),
        prefix="country_cat",
        drop_first=True,
        dtype=int,
    )
    ecoli = pd.get_dummies(
        df["wq27_decile"].astype("Int64"),
        prefix="wq27_d",
        drop_first=True,
        dtype=int,
    )
    df = pd.concat([df, ws, country, ecoli], axis=1)

    for column in ["Any_U5", "Girls_less_than15", "Boys_15or_less"]:
        df[column] = df[column].fillna(0).astype("int8")

    for value, label in {
        1: "flush",
        2: "pit_latrine",
        3: "open_defecation",
        98: "missing",
    }.items():
        df[f"toilet_{label}"] = (df["Toilet"] == value).astype("int8")

    any_risk = df["RiskHome"] == 1
    very_high_risk = df["RiskHome"] == 2

    df["SomeRiskHome"] = any_risk.astype("int8")
    df["VeryHighRiskHome"] = very_high_risk.astype("int8")

    return df


### Treatment variables

When households report multiple methods, the categorical treatment follows this priority: boiling, chlorine/tablets, physical treatment, and residual other/solar. The physical-treatment category combines household filters, cloth straining, and settling and is labelled **Straining/settling** in the results. Residual other/solar observations remain treated in the binary IRM analysis but are excluded from method-specific APOS analyses.


In [ ]:
def treatment(df):
    '''Create binary and categorical treatment variables.'''

    method_cols = [
        "WQ15A",
        "WQ15B",
        "WQ15C",
        "WQ15D",
        "WQ15E",
        "WQ15F",
        "WQ15G",
        "WQ15H",
        "WQ15X",
    ]

    missing = sorted(set(method_cols) - set(df.columns))

    if missing:
        raise KeyError(f"Missing water-treatment indicators: {missing}")

    used = pd.DataFrame(
        {
            column: df[column].astype("string").str.strip().eq(column[-1])
            for column in method_cols
        },
        index=df.index,
    )
    df["treatment_count"] = used.sum(axis=1).astype(int)
    df["single_method"] = (df["treatment_count"] == 1).astype("int8")
    df["water_treatment"] = used.any(axis=1).astype("int8")

    category = pd.Series(0, index=df.index, dtype="int8")
    category[used["WQ15E"] | used["WQ15X"]] = 4
    category[used["WQ15C"] | used["WQ15D"] | used["WQ15F"]] = 3
    category[used["WQ15B"] | used["WQ15G"] | used["WQ15H"]] = 2
    category[used["WQ15A"]] = 1

    df["treat_boil"] = category.eq(1).astype("int8")
    df["treat_chlorine"] = category.eq(2).astype("int8")
    df["treat_straining_settling"] = category.eq(3).astype("int8")
    df["treat_other"] = category.eq(4).astype("int8")
    df["treat_cat"] = category.mask(category.eq(4)).astype("Int8")

    return df


### Child variables


In [ ]:
def prep(raw, child=False):
    '''Prepare one analysis sample.'''

    df = treatment(base(raw))

    if not child:
        return df

    needed = ["age", "male", "HHID", "diarrhea"]
    missing = sorted(set(needed) - set(df.columns))

    if missing:
        raise KeyError(f"Missing U5 columns: {missing}")

    df["child_age"] = df["age"].astype(int)
    age = pd.get_dummies(
        df["child_age"],
        prefix="child_age",
        drop_first=True,
        dtype=int,
    )
    df = pd.concat([df, age], axis=1)
    df["child_sex_male"] = df["male"].astype("int8")
    df["diarrhea"] = df["diarrhea"].astype("Int64")

    return df


### Load samples


In [ ]:
hh_raw, _ = pyreadstat.read_dta(HH_FILE)
u5_raw, _ = pyreadstat.read_dta(U5_FILE)

hh = prep(hh_raw)
u5 = prep(u5_raw, child=True)


## Robust specification for major overlap

This specification keeps only countries with at least 10 households in each of the five treatment levels. Country eligibility is defined with the HH sample and the same countries are used for HH and U5.


In [ ]:
original_rows = {"HH": len(hh), "U5": len(u5)}
minimum_per_level = 10

coverage = pd.crosstab(hh["country_cat"], hh["treat_cat"]).reindex(
    columns=list(LEVELS),
    fill_value=0,
)
eligible_countries = coverage.index[
    coverage.ge(minimum_per_level).all(axis=1)
]

if eligible_countries.empty:
    raise ValueError("No country has complete treatment coverage.")

if "Country" not in hh.columns:
    raise KeyError("Country names are missing from the HH data.")

country_names = hh[["country_cat", "Country"]].drop_duplicates()

if country_names["country_cat"].duplicated().any():
    raise ValueError("A country code maps to more than one country name.")

country_status = (
    coverage.rename(columns=LEVELS)
    .reset_index()
    .merge(country_names, on="country_cat", validate="one_to_one")
)
country_status["status"] = np.where(
    country_status["country_cat"].isin(eligible_countries),
    "Kept",
    "Dropped",
)
country_status["levels_below_minimum"] = [
    ", ".join(
        LEVELS[level]
        for level in LEVELS
        if row[LEVELS[level]] < minimum_per_level
    )
    for _, row in country_status.iterrows()
]
country_status = country_status[
    [
        "status",
        "Country",
        "country_cat",
        "levels_below_minimum",
        *LEVELS.values(),
    ]
]

restricted = {}

for dataset, frame in [("HH", hh), ("U5", u5)]:
    selected = frame[frame["country_cat"].isin(eligible_countries)].copy()
    country_columns = [
        column for column in selected.columns
        if column.startswith("country_cat_")
    ]
    selected = selected.drop(columns=country_columns)
    country_dummies = pd.get_dummies(
        selected["country_cat"].astype("Int64"),
        prefix="country_cat",
        drop_first=True,
        dtype=int,
    )
    restricted[dataset] = pd.concat([selected, country_dummies], axis=1)

hh = restricted["HH"]
u5 = restricted["U5"]

for dataset, frame in [("HH", hh), ("U5", u5)]:
    counts = frame["treat_cat"].value_counts().reindex(LEVELS, fill_value=0)

    if counts.eq(0).any():
        raise ValueError(f"A treatment level is absent from the restricted {dataset} sample.")

summary = pd.DataFrame(
    [
        {
            "dataset": "HH",
            "countries": hh["country_cat"].nunique(),
            "kept": len(hh),
            "original": original_rows["HH"],
            "dropped": original_rows["HH"] - len(hh),
        },
        {
            "dataset": "U5",
            "countries": u5["country_cat"].nunique(),
            "kept": len(u5),
            "original": original_rows["U5"],
            "dropped": original_rows["U5"] - len(u5),
        },
    ]
)

display(summary)
display(Markdown("### Countries kept"))
display(
    country_status[country_status["status"] == "Kept"]
    .sort_values("Country")
    .reset_index(drop=True)
)
display(Markdown("### Countries dropped"))
display(
    country_status[country_status["status"] == "Dropped"]
    .sort_values("Country")
    .reset_index(drop=True)
)
country_status.to_csv(
    OUT / "countries_major_overlap_notebook.csv",
    index=False,
)


### Sample summary

Outcome means below are unconditional means for the full IRM analysis sample. They are not means restricted to untreated households. The DDML table footers use this same definition.


In [ ]:
sample_summary = pd.DataFrame(
    [
        {
            "dataset": "HH",
            "rows": len(hh),
            "treatment_rate": hh["water_treatment"].mean(),
            "SomeRiskHome": hh["SomeRiskHome"].mean(),
            "VeryHighRiskHome": hh["VeryHighRiskHome"].mean(),
        },
        {
            "dataset": "U5",
            "rows": len(u5),
            "treatment_rate": u5["water_treatment"].mean(),
            "SomeRiskHome": u5["SomeRiskHome"].mean(),
            "VeryHighRiskHome": u5["VeryHighRiskHome"].mean(),
            "diarrhea": u5["diarrhea"].mean(),
            "clusters": u5["HHID"].nunique(),
        },
    ]
)

display(sample_summary)


## Model specification

Education is excluded from the control vector because its source categories are not comparable across countries. The prior leave-one-control-out diagnostic showed only small changes in the DDML estimates when education was removed.

### Control groups


In [ ]:
BASE_X = {
    "wealth": ["wealth_q2", "wealth_q3", "wealth_q4", "wealth_q5"],
    "country": ["country_cat"],
    "urban": ["urban_bin"],
    "water_source": ["water_source"],
    "household": ["Any_U5", "Girls_less_than15", "Boys_15or_less"],
    "sanitation": [
        "toilet_pit_latrine",
        "toilet_open_defecation",
        "toilet_missing",
    ],
    "source_ecoli": ["wq27_decile"],
}

U5_X = {
    **BASE_X,
    "child": ["child_age", "child_sex_male"],
}


In [ ]:
def controls(df, groups):
    '''Expand grouped control names into model columns.'''

    prefixes = {
        "water_source": "ws1g_",
        "country_cat": "country_cat_",
        "wq27_decile": "wq27_d_",
        "child_age": "child_age_",
    }
    columns = []
    grouped = {}

    for group, names in groups.items():
        expanded_group = []

        for name in names:
            if name in prefixes:
                expanded = sorted(
                    [column for column in df.columns if column.startswith(prefixes[name])],
                    key=lambda column: (len(column), column),
                )
                if name == "water_source":
                    expanded = expanded[1:]
            else:
                expanded = [name]

            columns.extend(expanded)
            expanded_group.extend(expanded)

        grouped[group] = list(dict.fromkeys(expanded_group))

    columns = list(dict.fromkeys(columns))
    missing = [column for column in columns if column not in df.columns]

    if missing:
        raise KeyError(f"Missing controls: {missing}")

    return columns, grouped


hh_x, hh_groups = controls(hh, BASE_X)
u5_x, u5_groups = controls(u5, U5_X)

display(
    pd.DataFrame(
        {
            "dataset": ["HH", "U5"],
            "controls": [len(hh_x), len(u5_x)],
        }
    )
)


### Outcome learners


In [ ]:
# Elastic Net uses bounded inner tuning; other learners use outer parallelism.
reg = {
    "ols": LinearRegression(),
    "lasso": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                LassoCV(
                    cv=3,
                    max_iter=5_000,
                    n_jobs=1,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "ridge": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                RidgeCV(
                    alphas=np.logspace(-3, 3, 7),
                    cv=3,
                ),
            ),
        ]
    ),
    "enet": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                ElasticNetCV(
                    cv=3,
                    l1_ratio=[0.5],
                    max_iter=5_000,
                    n_jobs=TUNING_WORKERS,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "rf": RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        random_state=SEED,
        n_jobs=1,
    ),
    "xgb": XGBRegressor(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        random_state=SEED,
        n_jobs=1,
        eval_metric="rmse",
    ),
}


### Propensity learners


In [ ]:
grid = np.logspace(-3, 3, 7)

clf = {
    "ols": LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=1_000,
    ),
    "lasso": LogisticRegressionCV(
        cv=3,
        Cs=grid,
        solver="liblinear",
        max_iter=1_000,
        n_jobs=1,
        random_state=SEED,
        scoring="roc_auc",
        l1_ratios=(1,),
    ),
    "ridge": LogisticRegressionCV(
        cv=3,
        Cs=grid,
        solver="lbfgs",
        max_iter=1_000,
        n_jobs=1,
        random_state=SEED,
        scoring="roc_auc",
        l1_ratios=(0,),
    ),
    "enet": LogisticRegressionCV(
        cv=3,
        Cs=np.logspace(-2, 2, 5),
        solver="saga",
        max_iter=2_000,
        n_jobs=TUNING_WORKERS,
        random_state=SEED,
        scoring="roc_auc",
        l1_ratios=[0.5],
    ),
    "rf": RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        random_state=SEED,
        n_jobs=1,
    ),
    "xgb": XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        random_state=SEED,
        n_jobs=1,
        eval_metric="logloss",
    ),
}


### Stacked learner


In [ ]:
learners = {
    name: {
        "g": clone(reg[name]),
        "m": clone(clf[name]),
    }
    for name in reg
}

learners["stacked"] = {
    "g": StackingRegressor(
        estimators=[
            (name, clone(model))
            for name, model in reg.items()
        ],
        final_estimator=RidgeCV(
            alphas=np.logspace(-3, 6, 10),
            cv=5,
        ),
        cv=3,
        n_jobs=1,
    ),
    "m": StackingClassifier(
        estimators=[
            (name, clone(model))
            for name, model in clf.items()
        ],
        final_estimator=clone(clf["ridge"]),
        cv=3,
        n_jobs=1,
    ),
}


### Estimation samples


In [ ]:
SPECS = [
    ("HH", hh, "SomeRiskHome", hh_x, hh_groups, None),
    ("HH", hh, "VeryHighRiskHome", hh_x, hh_groups, None),
    ("U5", u5, "diarrhea", u5_x, u5_groups, "HHID"),
]


### Model checkpoints


In [ ]:
def fit_sig(
    kind,
    data_sig,
    outcome,
    treatment,
    controls,
    learner,
    cluster=None,
):
    """Identify the exact data and model specification used by one fit."""

    settings = {
        "kind": kind,
        "data": data_sig,
        "outcome": outcome,
        "treatment": treatment,
        "controls": controls,
        "cluster": cluster,
        "ml_g": joblib_hash(learner["g"]),
        "ml_m": joblib_hash(learner["m"]),
        "folds": FOLDS,
        "repetitions": IRM_REPS if kind == "IRM" else APOS_REPS,
        "trimming": TRIM,
        "levels": list(LEVELS) if kind == "APOS" else None,
        "sensitivity": {
            "cf_y": 0.03,
            "cf_d": 0.03,
            "rho": 1.0,
            "level": 0.95,
        },
        "seed": SEED,
        "doubleml": dml.__version__,
        "sklearn": sklearn.__version__,
    }

    return joblib_hash(settings)


def old_sig(cache, kind, data_sig, sample, outcome, treatment, cluster=None):
    """Validate and identify a checkpoint created by the previous notebook."""

    required = {"model", "sample", "controls"}

    if not required.issubset(cache):
        return None

    if not cache["sample"].equals(sample):
        return None

    model = cache["model"]

    if model.n_folds != FOLDS:
        return None

    expected_reps = IRM_REPS if kind == "IRM" else APOS_REPS

    if model.n_rep != expected_reps:
        return None

    if float(model._trimming_threshold) != TRIM:
        return None

    if kind == "IRM" and model.score != "ATE":
        return None

    if kind == "APOS" and list(model.treatment_levels) != list(LEVELS):
        return None

    learner = {
        "g": model.learner["ml_g"],
        "m": model.learner["ml_m"],
    }

    return fit_sig(
        kind,
        data_sig,
        outcome,
        treatment,
        cache["controls"],
        learner,
        cluster,
    )


In [ ]:
def save_fit(path, cache):
    """Write a checkpoint atomically."""

    temporary = path.with_suffix(".tmp")

    with temporary.open("wb") as file:
        pickle.dump(cache, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())

    os.replace(temporary, path)


def load_fit(
    path,
    signature,
    kind,
    data_sig,
    sample,
    outcome,
    treatment,
    cluster=None,
):
    """Load only a checkpoint that matches the current specification."""

    if not path.is_file():
        return None

    try:
        with path.open("rb") as file:
            cache = pickle.load(file)
    except Exception as error:
        raise RuntimeError(f"Cannot read checkpoint {path.name}.") from error

    if cache.get("signature") == signature:
        tqdm.write(f"Loaded {path.name}")
        return cache

    legacy_signature = old_sig(
        cache,
        kind,
        data_sig,
        sample,
        outcome,
        treatment,
        cluster,
    )

    if legacy_signature == signature:
        compact = {
            "signature": signature,
            "model": cache["model"],
        }

        if kind == "APOS":
            compact["contrast"] = cache["contrast"]

        save_fit(path, compact)
        tqdm.write(f"Migrated and loaded {path.name}")
        return compact

    tqdm.write(f"Checkpoint does not match; refitting {path.name}")
    return None


### Publication tables


In [ ]:
OUTCOME_LABELS = {
    "SomeRiskHome": r"Some Risk Home (E.coli 1--100 CFU)",
    "VeryHighRiskHome": r"Very High Risk Home (E.coli $\geq$ 101 CFU)",
    "diarrhea": "Diarrhea (under-5)",
}

LEARNER_LABELS = {
    "ols": "OLS",
    "lasso": "Lasso",
    "ridge": "Ridge",
    "enet": "Elastic Net",
    "rf": "Random Forest",
    "xgb": "XGBoost",
    "stacked": "Stacked",
}

GROUP_LABELS = {
    "RiskSource": "Source-water E.coli risk",
    "windex5": "Wealth quintile",
    "country_cat": "Country",
    "Toilet": "Toilet type",
    "WS1_g": "Water source type",
    "num_children_bin": "Number of children",
    "child_age": "Child age",
}

GROUP_FILES = {
    "RiskSource": "risksource",
    "windex5": "windex5",
    "country_cat": "countrycat",
    "Toilet": "toilet",
    "WS1_g": "ws1g",
    "num_children_bin": "numchildrenbin",
    "child_age": "childage",
}

GROUP_VALUES = {
    "RiskSource": {0: "No risk", 1: "Moderate", 2: "Very high"},
    "windex5": {1: "Q1", 2: "Q2", 3: "Q3", 4: "Q4", 5: "Q5"},
    "Toilet": {1: "Flush", 2: "Pit latrine", 3: "Open defecation", 98: "Missing"},
    "WS1_g": {
        11: "Piped",
        21: "Tube/well/borehole",
        31: "Protected well/spring",
        32: "Unprotected well/spring",
        51: "Surface/rain",
        91: "Packaged/bottled",
        96: "Other",
    },
    "num_children_bin": {0: "0", 1: "1", 2: "2", 3: "3", 4: "4+"},
}


def stars(coef, se):
    """Return normal-approximation significance stars."""

    if pd.isna(coef) or pd.isna(se) or se <= 0:
        return ""

    z = abs(coef / se)

    if z > 2.576:
        return "***"
    if z > 1.960:
        return "**"
    if z > 1.645:
        return "*"
    return ""


def estimate(coef, se):
    """Format a coefficient and standard error for a LaTeX cell."""

    return f"{coef:.3f}{stars(coef, se)} ({se:.3f})"


def group_label(variable, value):
    """Return the publication label for one group value."""

    value = int(value)
    return GROUP_VALUES.get(variable, {}).get(value, str(value))


def write_tex(lines, filename):
    """Write the same publication table to the output and manuscript folders."""

    path = Path(filename)
    filename = f"{path.stem}_major_overlap{path.suffix}"
    text = "\n".join(lines) + "\n"

    for folder in [OUT, TABLES]:
        (folder / filename).write_text(text, encoding="utf-8")

    print(f"LaTeX table: {filename}")


In [ ]:
def results_tex(results, outcomes, caption, label, filename, landscape):
    """Create the publication table for IRM and APOS estimates."""

    learners = LEARNER_NAMES
    treatments = ["Any Treatment", "Boil", "Chlorine", "Straining/settling"]
    ncols = 1 + len(outcomes) * len(learners)
    column_spec = "l|" + "|".join("c" * len(learners) for _ in outcomes)

    def span(body, index):
        spec = "c" if index == len(outcomes) - 1 else "c|"
        return rf"\multicolumn{{{len(learners)}}}{{{spec}}}{{{body}}}"

    lines = [
        r"% Requires: \usepackage{booktabs, pdflscape, graphicx, adjustbox}",
    ]

    if landscape:
        lines.append(r"\begin{landscape}")

    max_height = r"0.70\textwidth" if landscape else r"0.86\textheight"
    lines += [
        r"\begin{table}[htbp]",
        r"\centering",
        rf"\caption{{Robust specification for major overlap: {caption}}}",
        rf"\label{{{label}}}",
        r"\scriptsize",
        r"\setlength{\tabcolsep}{3pt}",
        r"\renewcommand{\arraystretch}{0.94}",
        rf"\begin{{adjustbox}}{{max width=\linewidth, max totalheight={max_height}, center}}",
        rf"\begin{{tabular}}{{{column_spec}}}",
        r"\toprule",
        " & ".join([""] + [span(OUTCOME_LABELS[outcome], i) for i, outcome in enumerate(outcomes)]) + r" \\",
        " ".join(
            rf"\cmidrule(lr){{{2 + i * len(learners)}-{1 + (i + 1) * len(learners)}}}"
            for i in range(len(outcomes))
        ),
        " & ".join(
            [""]
            + [LEARNER_LABELS[learner] for _ in outcomes for learner in learners]
        )
        + r" \\",
        r"\midrule",
    ]

    for treatment_index, treatment in enumerate(treatments):
        coef_row = [treatment]
        se_row = [""]
        rv_row = [r"\quad RV"]
        rva_row = [r"\quad RV$_\alpha$"]

        for outcome in outcomes:
            for learner in learners:
                hit = results[
                    (results["outcome"] == outcome)
                    & (results["treatment"] == treatment)
                    & (results["learner"] == learner)
                ]

                if len(hit) != 1:
                    raise ValueError(
                        f"Expected one result for {outcome}, {treatment}, {learner}."
                    )

                row = hit.iloc[0]
                coef_row.append(f"{row['coef']:.3f}{stars(row['coef'], row['se'])}")
                se_row.append(f"({row['se']:.3f})")
                rv_row.append(f"{row['rv']:.2f}")
                rva_row.append(f"{row['rva']:.2f}")

        lines += [
            " & ".join(coef_row) + r" \\",
            " & ".join(se_row) + r" \\",
            " & ".join(rv_row) + r" \\",
            " & ".join(rva_row) + r" \\",
        ]

        n_row = [r"\quad $N$"]
        for i, outcome in enumerate(outcomes):
            sample_rows = results[
                (results["outcome"] == outcome)
                & (results["treatment"] == treatment)
            ]
            n = int(sample_rows["n"].iloc[0])
            n_row.append(span(f"{n:,}", i))
        lines.append(" & ".join(n_row) + r" \\")

        if treatment_index < len(treatments) - 1:
            lines.append(r"\midrule")

    lines.append(r"\midrule")

    for title, column, formatter in [
        (r"Outcome mean $\bar Y$", "y_mean", lambda value: f"{value:.3f}"),
        ("Treated (\\%)", "treatment_rate", lambda value: f"{100 * value:.1f}"),
        ("Clusters", "clusters", lambda value: "---" if pd.isna(value) else f"{int(value):,}"),
    ]:
        row = [title]
        for i, outcome in enumerate(outcomes):
            full_sample = results[
                (results["outcome"] == outcome)
                & (results["treatment"] == "Any Treatment")
            ]
            value = full_sample[column].iloc[0]
            row.append(span(formatter(value), i))
        lines.append(" & ".join(row) + r" \\")

    notes = (
        r"\textit{Notes:} Each coefficient is reported with significance stars over "
        r"its standard error. Outcome means are unconditional means from the full "
        r"IRM analysis sample. Any treatment is the IRM ATE of any household water "
        r"treatment versus none. Boil, chlorine, and straining/settling are APOS contrasts "
        r"relative to no treatment. RV and RV$_\alpha$ are the robustness value and "
        r"its 95\% confidence-interval-adjusted version. All models control for wealth, "
        r"country fixed effects, urban residence, water source, household "
        r"composition, sanitation, and source-water E.coli decile; the under-5 model "
        r"also controls for child age and sex. IRM cross-fitting uses 5 folds $\times$ "
        r"3 repetitions; APOS uses 5 folds $\times$ 1 repetition. Propensities are "
        r"trimmed to $[0.01,0.99]$. IRM standard errors cluster by HHID for diarrhea; "
        r"APOS contrasts use i.i.d. standard errors. Straining/settling combines "
        r"household filters, cloth straining, and settling. Residual other/solar "
        r"observations remain in the binary IRM but are excluded from APOS. "
        r"$^{***}p<0.01$, $^{**}p<0.05$, $^{*}p<0.10$."
    )

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{adjustbox}",
        r"\par\vspace{3pt}",
        rf"\begin{{minipage}}{{\linewidth}}\scriptsize {notes}\end{{minipage}}",
        r"\end{table}",
    ]

    if landscape:
        lines.append(r"\end{landscape}")

    write_tex(lines, filename)


In [ ]:
def group_tex(results, variable, value, kind):
    """Create one panelled GATE or GAPO table for a grouping variable."""

    subset = results[results["group_variable"] == variable].copy()

    if subset.empty:
        raise ValueError(f"No {kind} results for {variable}.")

    outcomes = list(subset[["dataset", "outcome"]].drop_duplicates().itertuples(index=False, name=None))
    groups = sorted(subset["group"].astype(int).unique())
    treatments = [LEVELS[level] for level in (list(LEVELS)[1:] if kind == "GATE" else LEVELS)]
    wide = len(groups) > 8
    ncols = len(groups) + 1
    title = "Group average treatment effects" if kind == "GATE" else "Group average potential outcomes"
    file_key = "source" if kind == "GAPO" and variable == "RiskSource" else GROUP_FILES[variable]
    filename = f"table_{kind.lower()}_{file_key}.tex"

    lines = [r"% Requires: \usepackage{booktabs, pdflscape, graphicx, adjustbox}"]

    if wide:
        lines.append(r"\begin{landscape}")

    lines += [
        r"\begin{table}[p]",
        r"\centering",
        rf"\caption{{Robust specification for major overlap: {title} by {GROUP_LABELS[variable].lower()}}}",
        rf"\label{{tab:major-overlap-{kind.lower()}_{GROUP_FILES[variable]}}}",
        r"\scriptsize" if wide else r"\small",
        r"\setlength{\tabcolsep}{4pt}",
    ]

    lines += [
        r"\begin{adjustbox}{max width=\linewidth, center}",
        r"\begin{tabular}{l" + "c" * len(groups) + "}",
        r"\toprule",
    ]

    for panel, (dataset, outcome) in enumerate(outcomes):
        data = subset[(subset["dataset"] == dataset) & (subset["outcome"] == outcome)]
        panel_label = chr(ord("A") + panel)
        lines += [
            rf"\multicolumn{{{ncols}}}{{l}}{{\textbf{{Panel {panel_label}: {OUTCOME_LABELS[outcome]} ({dataset})}}}} \\",
            ("Method vs. no treatment" if kind == "GATE" else "Treatment level")
            + " & "
            + " & ".join(group_label(variable, group) for group in groups)
            + r" \\",
            r"\midrule",
        ]

        for treatment in treatments:
            row = [treatment]

            for group in groups:
                hit = data[
                    (data["treatment"] == treatment)
                    & (data["group"].astype(int) == group)
                ]

                if len(hit) != 1:
                    raise ValueError(
                        f"Expected one {kind} result for {outcome}, {variable}, {group}, {treatment}."
                    )

                result = hit.iloc[0]

                if kind == "GATE":
                    row.append(estimate(result[value], result["se"]))
                else:
                    low = result[value] - 1.96 * result["se"]
                    high = result[value] + 1.96 * result["se"]
                    row.append(
                        rf"{result[value]:.3f} {{\scriptsize$[{low:.3f},{high:.3f}]$}}"
                    )

            lines.append(" & ".join(row) + r" \\")

        if panel < len(outcomes) - 1:
            lines.append(r"\addlinespace")

    if kind == "GATE":
        notes = (
            r"\textit{Notes:} Cells report APOS-based group average treatment effects "
            r"relative to no treatment. Standard errors are in parentheses. "
            r"$^{***}p<0.01$, $^{**}p<0.05$, $^{*}p<0.10$."
        )
    else:
        notes = (
            r"\textit{Notes:} Cells report APOS-based group average potential outcomes. "
            r"Brackets contain pointwise 95\% confidence intervals."
        )

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{adjustbox}",
    ]

    lines += [
        r"\par\vspace{3pt}",
        rf"\begin{{minipage}}{{\linewidth}}\footnotesize {notes}\end{{minipage}}",
        r"\end{table}",
    ]

    if wide:
        lines.append(r"\end{landscape}")

    write_tex(lines, filename)


In [ ]:
def gapo_tex(results):
    """Create one multipage GAPO table with sections by grouping variable."""

    treatments = [LEVELS[level] for level in LEVELS]
    outcomes = list(
        results[["dataset", "outcome"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
    variables = [
        variable
        for variable in GROUP_VARIABLES + ["child_age"]
        if variable in results["group_variable"].unique()
    ]

    lines = [
        r"% Requires: \usepackage{booktabs, longtable, pdflscape}",
        r"\begin{landscape}",
        r"\scriptsize",
        r"\setlength{\tabcolsep}{5pt}",
        r"\renewcommand{\arraystretch}{0.95}",
        r"\begin{longtable}{lcccc}",
        r"\caption{Robust specification for major overlap: Group average potential outcomes by grouping variable}",
        r"\label{tab:major-overlap-gapo} \\",
        r"\toprule",
        r"Group & No treatment & Boil & Chlorine & Straining/settling \\",
        r"\midrule",
        r"\endfirsthead",
        r"\multicolumn{5}{l}{\textit{Table \thetable\ continued}} \\",
        r"\toprule",
        r"Group & No treatment & Boil & Chlorine & Straining/settling \\",
        r"\midrule",
        r"\endhead",
        r"\midrule",
        r"\multicolumn{5}{r}{\textit{Continued on next page}} \\",
        r"\endfoot",
        r"\bottomrule",
        r"\endlastfoot",
    ]

    for outcome_index, (dataset, outcome) in enumerate(outcomes):
        outcome_data = results[
            (results["dataset"] == dataset)
            & (results["outcome"] == outcome)
        ]
        panel = chr(ord("A") + outcome_index)
        lines.append(
            rf"\multicolumn{{5}}{{l}}{{\textbf{{Panel {panel}: {OUTCOME_LABELS[outcome]} ({dataset})}}}} \\"
        )

        outcome_variables = [
            variable
            for variable in variables
            if variable in outcome_data["group_variable"].unique()
        ]

        for variable_index, variable in enumerate(outcome_variables):
            variable_data = outcome_data[
                outcome_data["group_variable"] == variable
            ]
            groups = sorted(variable_data["group"].astype(int).unique())
            lines.append(
                rf"\multicolumn{{5}}{{l}}{{\textit{{{GROUP_LABELS[variable]}}}}} \\"
            )

            for group in groups:
                row = [group_label(variable, group)]

                for treatment in treatments:
                    hit = variable_data[
                        (variable_data["group"].astype(int) == group)
                        & (variable_data["treatment"] == treatment)
                    ]

                    if len(hit) != 1:
                        raise ValueError(
                            f"Expected one GAPO result for {outcome}, {variable}, "
                            f"{group}, {treatment}."
                        )

                    result = hit.iloc[0]
                    low = result["gapo"] - 1.96 * result["se"]
                    high = result["gapo"] + 1.96 * result["se"]
                    row.append(
                        rf"{result['gapo']:.3f} {{\scriptsize$[{low:.3f},{high:.3f}]$}}"
                    )

                lines.append(" & ".join(row) + r" \\")

            if variable_index < len(outcome_variables) - 1:
                lines.append(r"\midrule")

        if outcome_index < len(outcomes) - 1:
            lines += [r"\addlinespace", r"\midrule"]

    notes = (
        r"\textit{Notes:} Cells report APOS-based group average potential outcomes. "
        r"Brackets contain pointwise 95\% confidence intervals. Horizontal divisions "
        r"separate the pre-treatment variables used to form groups. APOS uses 5-fold "
        r"cross-fitting with one repetition and generalized propensity scores trimmed "
        r"to $[0.01,0.99]$. Straining/settling combines household filters, cloth "
        r"straining, and settling; residual other/solar observations are excluded."
    )
    lines += [
        r"\end{longtable}",
        r"\par\vspace{3pt}",
        rf"\begin{{minipage}}{{\linewidth}}\footnotesize {notes}\end{{minipage}}",
        r"\end{landscape}",
    ]
    write_tex(lines, "table_gapo.tex")


In [ ]:
def overlap_tex(results):
    """Create the compact overlap table for the stacked propensity learner."""

    data = results[results["learner"] == "stacked"].copy()
    lines = [
        r"% Requires: \usepackage{booktabs}",
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Robust specification for major overlap: Propensity-score overlap diagnostics for the stacked APOS learner}",
        r"\label{tab:major-overlap-overlap-apos}",
        r"\small\setlength{\tabcolsep}{5pt}",
        r"\begin{adjustbox}{max width=\linewidth, center}",
        r"\begin{tabular}{lllrrrrrr}",
        r"\toprule",
        r"Dataset & Outcome & Treatment & Min. & P1 & Median & P99 & Max. & Outside trim (\%) \\",
        r"\midrule",
    ]

    for row in data.sort_values(["dataset", "outcome", "level"]).itertuples():
        lines.append(
            f"{row.dataset} & {OUTCOME_LABELS[row.outcome]} & {row.treatment} & "
            f"{row.min:.3f} & {row.p01:.3f} & {row.median:.3f} & "
            f"{row.p99:.3f} & {row.max:.3f} & {100 * row.outside_trim:.1f} \\\\"
        )

    notes = (
        r"\textit{Notes:} Entries summarize cross-fitted generalized propensity scores. "
        r"Outside trim is the percentage below 0.01 or above 0.99. The CSV retains "
        r"diagnostics for every learner; this publication table reports the stacked learner."
    )
    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{adjustbox}",
        r"\par\vspace{3pt}",
        rf"\begin{{minipage}}{{\linewidth}}\footnotesize {notes}\end{{minipage}}",
        r"\end{table}",
    ]
    write_tex(lines, "table_overlap_notebook.tex")


CONTROL_LABELS = {
    "wealth": "Wealth quintile",
    "country": "Country fixed effects",
    "urban": "Urban residence",
    "water_source": "Water source",
    "household": "Household demographics",
    "sanitation": "Sanitation",
    "source_ecoli": "Source-water E.coli",
    "child": "Child age and sex",
}


def contrast_gain(long, short, outcome_variance):
    """Calculate confounder benchmarks for an APOS treatment contrast."""

    long_sigma = np.squeeze(long.sensitivity_elements["sigma2"], axis=0)
    long_nu = np.squeeze(long.sensitivity_elements["nu2"], axis=0)
    short_sigma = np.squeeze(short.sensitivity_elements["sigma2"], axis=0)
    short_nu = np.squeeze(short.sensitivity_elements["nu2"], axis=0)

    long_r2 = 1 - long_sigma / outcome_variance
    short_r2 = 1 - short_sigma / outcome_variance
    riesz_ratio = short_nu / long_nu

    cf_y = np.clip((long_r2 - short_r2) / (1 - long_r2), 0, 1)
    cf_d = np.clip((1 - riesz_ratio) / riesz_ratio, 0, 1)
    delta = np.asarray(short.all_thetas) - np.asarray(long.all_thetas)

    outcome_gain = short_sigma - long_sigma
    riesz_gain = long_nu - short_nu
    denominator = np.sqrt(
        np.multiply(
            outcome_gain,
            riesz_gain,
            out=np.zeros_like(outcome_gain),
            where=(outcome_gain > 0) & (riesz_gain > 0),
        )
    )
    rho = np.clip(
        np.divide(
            np.abs(delta),
            denominator,
            out=np.ones_like(delta),
            where=denominator != 0,
        ),
        0,
        1,
    ) * np.sign(delta)

    return {
        "cf_y": np.median(cf_y, axis=1),
        "cf_d": np.median(cf_d, axis=1),
        "rho": np.median(rho, axis=1),
        "delta_theta": np.median(delta, axis=1),
    }


def sensitivity_tex(results):
    """Create the confounder-benchmarking table used in the original analysis."""

    outcomes = [
        outcome
        for outcome in ["SomeRiskHome", "VeryHighRiskHome", "diarrhea"]
        if outcome in results["outcome"].unique()
    ]
    controls_order = list(CONTROL_LABELS)
    treatments = [
        ("Panel A: Any treatment", ["Any Treatment"]),
        ("Panel B: Specific treatments vs. no treatment", ["Boil", "Chlorine", "Straining/settling"]),
    ]
    columns = "l|" + "|".join("ccc" for _ in outcomes)
    total_columns = 1 + 3 * len(outcomes)

    lines = [
        r"% Requires: \usepackage{booktabs, pdflscape, adjustbox}",
        r"\begin{landscape}",
        r"\begin{table}[p]",
        r"\centering",
        r"\caption{Robust specification for major overlap: Sensitivity benchmarking: observed control groups and robustness values}",
        r"\label{tab:major-overlap-sensitivity-benchmark}",
        r"\scriptsize\setlength{\tabcolsep}{3pt}",
        r"\renewcommand{\arraystretch}{0.92}",
        r"\begin{adjustbox}{max width=\linewidth, max totalheight=0.72\textwidth, center}",
        rf"\begin{{tabular}}{{{columns}}}",
        r"\toprule",
    ]

    header = [""]
    for outcome in outcomes:
        header.append(rf"\multicolumn{{3}}{{c}}{{{OUTCOME_LABELS[outcome]}}}")
    lines.append(" & ".join(header) + r" \\")
    lines.append(
        " ".join(
            rf"\cmidrule(lr){{{2 + 3 * index}-{4 + 3 * index}}}"
            for index in range(len(outcomes))
        )
    )
    lines.append(
        " & ".join([""] + [r"cf$_y$ & cf$_d$ & $\Delta\theta$"] * len(outcomes))
        + r" \\"
    )
    lines.append(r"\midrule")

    for panel_index, (panel, panel_treatments) in enumerate(treatments):
        lines.append(rf"\multicolumn{{{total_columns}}}{{l}}{{\textit{{{panel}}}}} \\")

        for treatment_index, treatment in enumerate(panel_treatments):
            treatment_data = results[results["treatment"] == treatment]
            header = [treatment]

            for outcome in outcomes:
                hit = treatment_data[treatment_data["outcome"] == outcome]

                if hit.empty:
                    header.append(r"\multicolumn{3}{c}{---}")
                    continue

                result = hit.iloc[0]
                ate = estimate(result["ate"], result["se"])
                rv = 100 * result["rv"]
                rva = 100 * result["rva"]
                header.append(
                    rf"\multicolumn{{3}}{{c}}{{{ate} {{\scriptsize[RV {rv:.1f}/{rva:.1f}]}}}}"
                )

            lines.append(" & ".join(header) + r" \\")

            for group in controls_order:
                row = [rf"\quad {CONTROL_LABELS[group]}"]
                has_value = False

                for outcome in outcomes:
                    hit = treatment_data[
                        (treatment_data["outcome"] == outcome)
                        & (treatment_data["control_group"] == group)
                    ]

                    if hit.empty:
                        row += ["", "", ""]
                    else:
                        result = hit.iloc[0]
                        has_value = True
                        row += [
                            f"{100 * result['cf_y']:.1f}",
                            f"{100 * result['cf_d']:.1f}",
                            f"{100 * result['delta_theta']:+.2f}",
                        ]

                if has_value:
                    lines.append(" & ".join(row) + r" \\")

            if treatment_index < len(panel_treatments) - 1:
                lines.append(r"\midrule")

        if panel_index < len(treatments) - 1:
            lines.append(r"\midrule")

    notes = (
        r"\textit{Notes:} Each benchmark removes the indicated observed control group "
        r"and compares the reduced specification with the complete specification. "
        r"cf$_y$ and cf$_d$ are partial $R^2$ values ($\times100$) in the outcome and "
        r"treatment equations. $\Delta\theta$ ($\times100$) is the change in the ATE. "
        r"RV/RV$_\alpha$ are the robustness values for the point estimate and its 95\% "
        r"confidence interval. $^{***}p<0.01$, $^{**}p<0.05$, $^{*}p<0.10$."
    )
    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{adjustbox}",
        r"\par\vspace{3pt}",
        rf"\begin{{minipage}}{{\linewidth}}\footnotesize {notes}\end{{minipage}}",
        r"\end{table}",
        r"\end{landscape}",
    ]
    write_tex(lines, "table_sensitivity_benchmark.tex")


In [ ]:
def feature_label(name):
    """Return the group and publication label for one predictor."""

    if name.startswith("country_cat_"):
        return None
    if name.startswith("wealth_q"):
        return "Wealth quintile", f"Q{name[-1]} (vs Q1)"
    if name == "urban_bin":
        return "Residence", "Urban"
    if name.startswith("ws1g_"):
        code = int(name.split("_")[-1])
        return "Water source", GROUP_VALUES["WS1_g"].get(code, str(code))
    if name == "Any_U5":
        return "Household composition", "Any under-5 child"
    if name == "Girls_less_than15":
        return "Household composition", r"Girl aged $<15$"
    if name == "Boys_15or_less":
        return "Household composition", r"Boy aged $\leq15$"
    if name.startswith("toilet_"):
        return "Sanitation", name.removeprefix("toilet_").replace("_", " ").title()
    if name.startswith("wq27_d_"):
        return r"Source E.\,coli", f"Decile {name.split('_')[-1]}"
    return "Other", name.replace("_", r"\_")


def predictor_tex(results, treatment):
    """Create the publication table for treatment-adoption predictors."""

    rows = []

    for result in results.itertuples():
        label = feature_label(result.name)

        if label is None:
            continue

        importance = (result.rf_importance + result.xgb_importance) / 2
        rows.append((importance, label[0], label[1], result))

    rows.sort(key=lambda item: item[0], reverse=True)
    lines = [
        r"% Requires: \usepackage{booktabs}",
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Robust specification for major overlap: Characteristics associated with water-treatment adoption}",
        r"\label{tab:major-overlap-treatment-predictors}",
        r"\small\setlength{\tabcolsep}{5pt}",
        r"\begin{tabular}{lccccc}",
        r"\toprule",
        r"Covariate & Logit & Lasso & E.\,net & RF & XGB \\",
        r" & (log-odds) & \multicolumn{2}{c}{(penalized coef.)} & \multicolumn{2}{c}{(importance $\times100$)} \\",
        r"\cmidrule(lr){3-4}\cmidrule(lr){5-6}",
        r"\midrule",
    ]

    for _, group, label, result in rows:
        p = result.logit_p
        marker = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""
        lines.append(
            rf"{label} \textit{{({group})}} & {result.logit_coef:+.2f}{marker} & "
            f"{result.lasso_coef:+.2f} & {result.enet_coef:+.2f} & "
            f"{100 * result.rf_importance:.1f} & {100 * result.xgb_importance:.1f} \\\\"
        )

    notes = (
        rf"\textit{{Notes:}} Outcome is any household water treatment "
        rf"($N={len(treatment):,}$, adoption rate ${treatment.mean():.3f}$). Rows are "
        r"ordered by mean random-forest/XGBoost importance. All models include country "
        r"fixed effects, which are omitted from the table. Reference categories are Q1 "
        r"for wealth, flush toilet, piped water, and rural residence. "
        r"Logit reports signed log-odds; lasso and elastic net report penalized coefficients; "
        r"RF and XGB report feature importance. "
        r"$^{***}p<0.01$, $^{**}p<0.05$, $^{*}p<0.10$."
    )
    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\par\vspace{3pt}",
        rf"\begin{{minipage}}{{\linewidth}}\footnotesize {notes}\end{{minipage}}",
        r"\end{table}",
    ]
    write_tex(lines, "table_treatment_predictors.tex")


## Main effects

### IRM: Any treatment

Cross-fitting is parallelized across folds. Each fitted bundle is saved immediately after estimation.


In [ ]:
irm = {}

for dataset, df, outcome, x, groups, cluster in tqdm(
    SPECS,
    desc="IRM outcomes",
):
    columns = [outcome, "water_treatment", *x]

    if cluster:
        columns.append(cluster)

    sample = (
        df[columns]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .reset_index(drop=True)
    )
    counts = sample["water_treatment"].value_counts()

    if len(counts) != 2 or counts.min() < 20:
        raise ValueError(f"Insufficient treatment variation: {counts.to_dict()}")

    data_sig = joblib_hash(sample)
    data = dml.DoubleMLData(
        sample,
        y_col=outcome,
        d_cols="water_treatment",
        x_cols=x,
        cluster_cols=cluster,
    )

    for learner in tqdm(
        LEARNER_NAMES,
        desc=f"{dataset} | {outcome}",
        leave=False,
    ):
        key = f"{dataset}_{outcome}_{learner}"
        path = MODELS / f"irm_{key}.pkl"
        signature = fit_sig(
            "IRM",
            data_sig,
            outcome,
            "water_treatment",
            x,
            learners[learner],
            cluster,
        )
        cache = load_fit(
            path,
            signature,
            "IRM",
            data_sig,
            sample,
            outcome,
            "water_treatment",
            cluster,
        )

        if cache is None:
            np.random.seed(SEED)
            model = dml.DoubleMLIRM(
                data,
                ml_g=clone(learners[learner]["g"]),
                ml_m=clone(learners[learner]["m"]),
                n_folds=FOLDS,
                n_rep=IRM_REPS,
                score="ATE",
                trimming_rule="truncate",
                trimming_threshold=TRIM,
            )
            model.fit(
                n_jobs_cv=IRM_WORKERS,
                store_models=False,
            )
            model.sensitivity_analysis(
                cf_y=0.03,
                cf_d=0.03,
                rho=1.0,
                level=0.95,
            )
            cache = {
                "signature": signature,
                "model": model,
            }
            save_fit(path, cache)

        irm[key] = {
            "model": cache["model"],
            "sample": sample,
            "controls": x,
            "groups": groups,
        }

print("IRM models:", len(irm))


### APOS: Treatment method

Treatment levels are fitted in parallel. Fold-level parallelism remains off here to prevent nested workers.


In [ ]:
apos = {}

for dataset, df, outcome, x, groups, cluster in tqdm(
    SPECS,
    desc="APOS outcomes",
):
    sample = (
        df[[outcome, "treat_cat", *x]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )
    rows = df.loc[sample.index].copy().reset_index(drop=True)
    sample["treat_cat"] = sample["treat_cat"].astype("int8")
    sample = sample.reset_index(drop=True)
    counts = sample["treat_cat"].value_counts().sort_index()

    if list(counts.index) != list(LEVELS):
        raise ValueError(f"Unexpected treatment levels: {counts.to_dict()}")

    if counts.min() < 50:
        raise ValueError(
            f"A treatment level has fewer than 50 observations: {counts.to_dict()}"
        )

    data_sig = joblib_hash(sample)
    data = dml.DoubleMLData(
        sample,
        y_col=outcome,
        d_cols="treat_cat",
        x_cols=x,
    )

    for learner in tqdm(
        LEARNER_NAMES,
        desc=f"{dataset} | {outcome}",
        leave=False,
    ):
        key = f"{dataset}_{outcome}_{learner}"
        path = MODELS / f"apos_{key}.pkl"
        signature = fit_sig(
            "APOS",
            data_sig,
            outcome,
            "treat_cat",
            x,
            learners[learner],
        )
        cache = load_fit(
            path,
            signature,
            "APOS",
            data_sig,
            sample,
            outcome,
            "treat_cat",
        )

        if cache is None:
            np.random.seed(SEED)
            model = dml.DoubleMLAPOS(
                data,
                ml_g=clone(learners[learner]["g"]),
                ml_m=clone(learners[learner]["m"]),
                treatment_levels=list(LEVELS),
                n_folds=FOLDS,
                n_rep=APOS_REPS,
                trimming_rule="truncate",
                trimming_threshold=TRIM,
            )
            model.fit(
                n_jobs_models=APOS_WORKERS,
                n_jobs_cv=1,
                store_models=False,
            )

            contrast = model.causal_contrast(reference_levels=0)
            contrast.sensitivity_analysis(cf_y=0.03, cf_d=0.03, rho=1.0)
            cache = {
                "signature": signature,
                "model": model,
                "contrast": contrast,
            }
            save_fit(path, cache)

        apos[key] = {
            "model": cache["model"],
            "contrast": cache["contrast"],
            "sample": sample,
            "rows": rows,
            "controls": x,
            "groups": groups,
        }

print("APOS models:", len(apos))


### Regression table


In [ ]:
main_rows = []

for key, bundle in irm.items():
    dataset, outcome, learner = key.split("_", 2)
    model = bundle["model"]
    ci = model.confint().iloc[0]
    sensitivity = model.sensitivity_params

    main_rows.append(
        {
            "method": "IRM",
            "dataset": dataset,
            "outcome": outcome,
            "treatment": "Any Treatment",
            "learner": learner,
            "coef": float(model.coef[0]),
            "se": float(model.se[0]),
            "ci_low": float(ci.iloc[0]),
            "ci_high": float(ci.iloc[1]),
            "n": len(bundle["sample"]),
            "y_mean": bundle["sample"][outcome].mean(),
            "treatment_rate": bundle["sample"]["water_treatment"].mean(),
            "clusters": (
                bundle["sample"]["HHID"].nunique()
                if "HHID" in bundle["sample"].columns
                else np.nan
            ),
            "rv": float(np.ravel(sensitivity["rv"])[0]),
            "rva": float(np.ravel(sensitivity["rva"])[0]),
        }
    )

for key, bundle in apos.items():
    dataset, outcome, learner = key.split("_", 2)
    summary = bundle["contrast"].summary.reset_index(drop=True)
    sensitivity = bundle["contrast"].sensitivity_params

    for row, level in enumerate(list(LEVELS)[1:]):
        result = summary.iloc[row]

        main_rows.append(
            {
                "method": "APOS",
                "dataset": dataset,
                "outcome": outcome,
                "treatment": LEVELS[level],
                "learner": learner,
                "coef": float(result["coef"]),
                "se": float(result["std err"]),
                "ci_low": float(result["2.5 %"]),
                "ci_high": float(result["97.5 %"]),
                "n": len(bundle["sample"]),
                "y_mean": bundle["sample"][outcome].mean(),
                "treatment_rate": (bundle["sample"]["treat_cat"] > 0).mean(),
                "clusters": (
                    bundle["rows"]["HHID"].nunique()
                    if "HHID" in bundle["rows"].columns
                    else np.nan
                ),
                "rv": float(np.ravel(sensitivity["rv"])[row]),
                "rva": float(np.ravel(sensitivity["rva"])[row]),
            }
        )

main = pd.DataFrame(main_rows)

main["estimate"] = [
    f"{coef:.3f} ({se:.3f})"
    for coef, se in zip(main["coef"], main["se"])
]

main_table = main.pivot(
    index=["method", "treatment"],
    columns=["outcome", "learner"],
    values="estimate",
)

display(Markdown("### Main regression results"))
display(main_table)

main.to_csv(OUT / "results_main_major_overlap_notebook.csv", index=False)

results_tex(
    main,
    ["SomeRiskHome", "VeryHighRiskHome"],
    "DDML estimates: Water treatment and household E.coli contamination",
    "tab:major-overlap-results-ecoli",
    "table_results_ecoli.tex",
    landscape=True,
)
results_tex(
    main,
    ["diarrhea"],
    "DDML estimates: Water treatment and child diarrhea (under-5)",
    "tab:major-overlap-results-diarrhea",
    "table_results_diarrhea.tex",
    landscape=False,
)


### Comparison with the main sample

The comparison uses the stacked learner because it is the headline specification.


In [ ]:
headline_file = OUT / "results_main_notebook.csv"

if not headline_file.is_file():
    raise FileNotFoundError(
        "Run the main notebook first to create results_main_notebook.csv."
    )

headline = pd.read_csv(headline_file)
keys = ["method", "dataset", "outcome", "treatment", "learner"]
comparison = (
    headline[headline["learner"] == "stacked"][keys + ["coef", "se"]]
    .merge(
        main[main["learner"] == "stacked"][keys + ["coef", "se"]],
        on=keys,
        suffixes=("_main", "_major_overlap"),
        validate="one_to_one",
    )
)
comparison["difference"] = comparison["coef_major_overlap"] - comparison["coef_main"]

display(comparison.set_index(keys).sort_index())
comparison.to_csv(
    OUT / "results_main_comparison_major_overlap_notebook.csv",
    index=False,
)


## Heterogeneity

### APOS signals


In [ ]:
GROUP_VARIABLES = [
    "RiskSource",
    "windex5",
    "country_cat",
    "Toilet",
    "WS1_g",
    "num_children_bin",
]

stacked_apos = {}

for key, bundle in apos.items():
    if not key.endswith("_stacked"):
        continue

    model = bundle["model"]
    rows = bundle["rows"]
    signals = {}

    for level, apo_model in zip(LEVELS, model.modellist):
        score = apo_model.psi_elements
        psi_a = np.asarray(score["psi_a"]).reshape(len(rows), -1)[:, 0]
        psi_b = np.asarray(score["psi_b"]).reshape(len(rows), -1)[:, 0]
        signals[level] = psi_b / (-psi_a)

    stacked_apos[key] = {
        "model": model,
        "rows": rows,
        "signals": signals,
    }


### Group average potential outcomes


In [ ]:
gapo_rows = []

for key, bundle in stacked_apos.items():
    dataset, outcome, learner = key.split("_", 2)
    group_variables = GROUP_VARIABLES + (["child_age"] if dataset == "U5" else [])

    for group_variable in group_variables:
        if group_variable not in bundle["rows"].columns:
            raise KeyError(f"Missing group variable: {group_variable}")

        values = bundle["rows"][group_variable]

        if values.isna().any():
            raise ValueError(f"Group variable {group_variable} contains missing group values.")

        if not values.eq(values.astype(int)).all():
            raise ValueError(f"Group variable {group_variable} must contain integer-coded groups.")

        groups = pd.get_dummies(
            values.astype(int),
            prefix="group",
            dtype=int,
        )

        for level, apo_model in zip(LEVELS, bundle["model"].modellist):
            for group, result in apo_model.gapo(groups).summary.iterrows():
                gapo_rows.append(
                    {
                        "dataset": dataset,
                        "outcome": outcome,
                        "group_variable": group_variable,
                        "group": int(str(group).removeprefix("group_")),
                        "level": level,
                        "treatment": LEVELS[level],
                        "gapo": float(result["coef"]),
                        "se": float(result["std err"]),
                    }
                )

gapo = pd.DataFrame(gapo_rows)


### Group average treatment effects


In [ ]:
gate_rows = []

for key, bundle in stacked_apos.items():
    dataset, outcome, learner = key.split("_", 2)
    group_variables = GROUP_VARIABLES + (["child_age"] if dataset == "U5" else [])

    for group_variable in group_variables:
        values = bundle["rows"][group_variable]

        if values.isna().any():
            raise ValueError(f"Group variable {group_variable} contains missing group values.")

        if not values.eq(values.astype(int)).all():
            raise ValueError(f"Group variable {group_variable} must contain integer-coded groups.")

        groups = pd.get_dummies(
            values.astype(int),
            prefix="group",
            dtype=float,
        )

        for level in list(LEVELS)[1:]:
            gate_model = dml.DoubleMLBLP(
                bundle["signals"][level] - bundle["signals"][0],
                basis=groups,
                is_gate=True,
            ).fit()

            for group, result in gate_model.summary.iterrows():
                gate_rows.append(
                    {
                        "dataset": dataset,
                        "outcome": outcome,
                        "group_variable": group_variable,
                        "group": int(str(group).removeprefix("group_")),
                        "level": level,
                        "treatment": LEVELS[level],
                        "coef": float(result["coef"]),
                        "se": float(result["std err"]),
                    }
                )

gate = pd.DataFrame(gate_rows)


### Heterogeneity tables


In [ ]:
gate["estimate"] = [
    f"{coef:.3f} ({se:.3f})"
    for coef, se in zip(gate["coef"], gate["se"])
]
gapo["estimate"] = [
    f"{coef:.3f} ({se:.3f})"
    for coef, se in zip(gapo["gapo"], gapo["se"])
]

gate_table = gate.pivot(
    index=["dataset", "outcome", "group_variable", "group"],
    columns="treatment",
    values="estimate",
).reindex(columns=[LEVELS[level] for level in list(LEVELS)[1:]])

gapo_table = gapo.pivot(
    index=["dataset", "outcome", "group_variable", "group"],
    columns="treatment",
    values="estimate",
).reindex(columns=[LEVELS[level] for level in LEVELS])

display(Markdown("#### Group average treatment effects"))
display(gate_table)
display(Markdown("#### Group average potential outcomes"))
display(gapo_table)

gate.to_csv(OUT / "results_gate_major_overlap_notebook.csv", index=False)
gapo.to_csv(OUT / "results_gapo_major_overlap_notebook.csv", index=False)

for group_variable in GROUP_VARIABLES + ["child_age"]:
    if group_variable in gate["group_variable"].unique():
        group_tex(gate, group_variable, "coef", "GATE")

gapo_tex(gapo)


## Diagnostics

### Propensity overlap


In [ ]:
overlap_rows = []

for key, bundle in apos.items():
    dataset, outcome, learner = key.split("_", 2)

    for level, apo_model in zip(LEVELS, bundle["model"].modellist):
        if "ml_m" not in apo_model.predictions:
            raise KeyError(f"Missing propensity predictions for {key}, level {level}.")

        propensity = np.asarray(
            apo_model.predictions["ml_m"],
            dtype=float,
        )
        propensity = propensity.reshape(propensity.shape[0], -1).mean(axis=1)

        overlap_rows.append(
            {
                "dataset": dataset,
                "outcome": outcome,
                "learner": learner,
                "level": level,
                "treatment": LEVELS[level],
                "min": propensity.min(),
                "p01": np.quantile(propensity, 0.01),
                "median": np.median(propensity),
                "p99": np.quantile(propensity, 0.99),
                "max": propensity.max(),
                "outside_trim": np.mean(
                    (propensity < TRIM) | (propensity > 1 - TRIM)
                ),
            }
        )

overlap = pd.DataFrame(overlap_rows)

overlap_table = overlap.set_index(
    ["dataset", "outcome", "treatment", "learner"]
).sort_index()

display(Markdown("### Propensity overlap diagnostics"))
display(overlap_table)

overlap.to_csv(OUT / "results_overlap_major_overlap_notebook.csv", index=False)
overlap_tex(overlap)


### IRM overlap plots


In [ ]:
for key, bundle in irm.items():
    if not key.endswith("_stacked"):
        continue

    dataset, outcome, learner = key.split("_", 2)
    propensity = np.asarray(
        bundle["model"].predictions["ml_m"],
        dtype=float,
    )
    propensity = propensity.reshape(propensity.shape[0], -1).mean(axis=1)
    treatment = bundle["sample"]["water_treatment"].to_numpy(dtype=int)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(
        propensity[treatment == 0],
        bins=40,
        density=True,
        alpha=0.55,
        label="No treatment",
    )
    ax.hist(
        propensity[treatment == 1],
        bins=40,
        density=True,
        alpha=0.55,
        label="Any treatment",
    )
    ax.axvline(TRIM, color="0.35", linestyle="--", linewidth=0.8)
    ax.axvline(1 - TRIM, color="0.35", linestyle="--", linewidth=0.8)
    ax.set(
        title=f"IRM propensity overlap | {dataset} | {outcome}",
        xlabel="Cross-fitted probability of any treatment",
        ylabel="Density",
        xlim=(0, 1),
    )
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(
        FIGS / f"overlap_irm_{dataset}_{outcome}_major_overlap_notebook.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


### APOS overlap plots


In [ ]:
from scipy.stats import gaussian_kde


grid = np.linspace(TRIM, 1 - TRIM, 300)

for key, bundle in apos.items():
    if not key.endswith("_stacked"):
        continue

    dataset, outcome, learner = key.split("_", 2)
    treatment = bundle["sample"]["treat_cat"].to_numpy(dtype=int)
    fig, axes = plt.subplots(1, len(LEVELS), figsize=(16, 3.2), sharey=True)

    for level, apo_model, ax in zip(
        LEVELS,
        bundle["model"].modellist,
        axes,
    ):
        propensity = np.asarray(
            apo_model.predictions["ml_m"],
            dtype=float,
        )
        propensity = propensity.reshape(propensity.shape[0], -1).mean(axis=1)

        for observed_level in LEVELS:
            values = propensity[treatment == observed_level]

            if len(values) < 2 or np.unique(values).size < 2:
                raise ValueError(
                    f"KDE requires variation for {dataset}, {outcome}, "
                    f"{LEVELS[level]} | observed {LEVELS[observed_level]}."
                )

            density = gaussian_kde(values)(grid)
            ax.plot(
                grid,
                density,
                linewidth=1.4,
                label=f"D = {LEVELS[observed_level]}",
            )
        ax.axvline(TRIM, color="0.35", linestyle="--", linewidth=0.8)
        ax.axvline(1 - TRIM, color="0.35", linestyle="--", linewidth=0.8)
        ax.set_title(LEVELS[level])
        ax.set_xlabel("Cross-fitted probability")
        ax.set_xlim(0, 1)

    axes[0].set_ylabel("Density")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=len(LEVELS),
        frameon=False,
        fontsize=8,
    )
    fig.suptitle(
        f"APOS propensity overlap | {dataset} | {outcome}",
        y=0.88,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.78))
    fig.savefig(
        FIGS / f"overlap_apos_{dataset}_{outcome}_major_overlap_notebook.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


### Sensitivity


In [ ]:
sens_rows = []
benchmark_rows = []

# IRM benchmarks use the full-sample any-treatment estimand.
for key, bundle in tqdm(
    [(key, value) for key, value in irm.items() if key.endswith("_stacked")],
    desc="IRM sensitivity benchmarks",
):
    dataset, outcome, learner = key.split("_", 2)
    model = bundle["model"]
    params = model.sensitivity_params

    if params is None:
        raise ValueError(f"Missing sensitivity results for {key}.")

    ate = float(np.ravel(model.coef)[0])
    se = float(np.ravel(model.se)[0])
    rv = float(np.ravel(params["rv"])[0])
    rva = float(np.ravel(params["rva"])[0])

    for group, group_controls in tqdm(
        bundle["groups"].items(),
        desc=f"{dataset} | {outcome} | IRM",
        leave=False,
    ):
        signature = joblib_hash(
            {
                "kind": "IRM sensitivity benchmark",
                "sample": joblib_hash(bundle["sample"]),
                "group": group,
                "removed_controls": group_controls,
                "learner": joblib_hash(learners["stacked"]),
                "folds": FOLDS,
                "repetitions": IRM_REPS,
                "seed": SEED,
                "doubleml": dml.__version__,
                "sklearn": sklearn.__version__,
            }
        )
        path = MODELS / f"sensitivity_irm_{dataset}_{outcome}_{group}.pkl"
        cached = None

        if path.is_file():
            with path.open("rb") as file:
                candidate = pickle.load(file)
            if candidate.get("signature") == signature:
                cached = candidate
                tqdm.write(f"Loaded {path.name}")

        if cached is None:
            result = model.sensitivity_benchmark(
                benchmarking_set=group_controls,
                fit_args={"n_jobs_cv": IRM_WORKERS, "store_models": False},
            ).iloc[0]
            cached = {
                "signature": signature,
                "benchmark": {
                    "cf_y": float(result["cf_y"]),
                    "cf_d": float(result["cf_d"]),
                    "rho": float(result["rho"]),
                    "delta_theta": float(result["delta_theta"]),
                },
            }
            save_fit(path, cached)

        benchmark_rows.append(
            {
                "method": "IRM",
                "dataset": dataset,
                "outcome": outcome,
                "treatment": "Any Treatment",
                "control_group": group,
                "ate": ate,
                "se": se,
                "rv": rv,
                "rva": rva,
                **cached["benchmark"],
            }
        )


# APOS benchmarks compare each treatment level with no treatment.
for key, bundle in tqdm(
    [(key, value) for key, value in apos.items() if key.endswith("_stacked")],
    desc="APOS sensitivity benchmarks",
):
    dataset, outcome, learner = key.split("_", 2)
    long_model = bundle["model"]
    long_contrast = bundle["contrast"]
    params = long_contrast.sensitivity_params

    if params is None:
        raise ValueError(f"Missing sensitivity results for {key}.")

    contrast_summary = long_contrast.summary.reset_index(drop=True)
    outcome_variance = float(np.var(bundle["sample"][outcome].to_numpy(dtype=float)))

    for row, level in enumerate(list(LEVELS)[1:]):
        sens_rows.append(
            {
                "dataset": dataset,
                "outcome": outcome,
                "level": level,
                "treatment": LEVELS[level],
                "rv": float(np.ravel(params["rv"])[row]),
                "rva": float(np.ravel(params["rva"])[row]),
            }
        )

    for group, group_controls in tqdm(
        bundle["groups"].items(),
        desc=f"{dataset} | {outcome} | APOS",
        leave=False,
    ):
        short_controls = [
            control for control in bundle["controls"]
            if control not in set(group_controls)
        ]
        signature = joblib_hash(
            {
                "kind": "APOS contrast sensitivity benchmark",
                "sample": joblib_hash(bundle["sample"]),
                "group": group,
                "removed_controls": group_controls,
                "remaining_controls": short_controls,
                "learner": joblib_hash(learners["stacked"]),
                "folds": FOLDS,
                "repetitions": APOS_REPS,
                "trimming": TRIM,
                "levels": list(LEVELS),
                "seed": SEED,
                "doubleml": dml.__version__,
                "sklearn": sklearn.__version__,
            }
        )
        path = MODELS / f"sensitivity_apos_{dataset}_{outcome}_{group}.pkl"
        cached = None

        if path.is_file():
            with path.open("rb") as file:
                candidate = pickle.load(file)
            if candidate.get("signature") == signature:
                cached = candidate
                tqdm.write(f"Loaded {path.name}")

        if cached is None:
            short_data = dml.DoubleMLData(
                bundle["sample"],
                y_col=outcome,
                d_cols="treat_cat",
                x_cols=short_controls,
            )
            short_model = dml.DoubleMLAPOS(
                short_data,
                ml_g=clone(learners["stacked"]["g"]),
                ml_m=clone(learners["stacked"]["m"]),
                treatment_levels=list(LEVELS),
                n_folds=FOLDS,
                n_rep=APOS_REPS,
                trimming_rule="truncate",
                trimming_threshold=TRIM,
            )
            short_model.set_sample_splitting(long_model.smpls)
            short_model.fit(
                n_jobs_models=APOS_WORKERS,
                n_jobs_cv=1,
                store_models=False,
            )
            short_contrast = short_model.causal_contrast(reference_levels=0)
            cached = {
                "signature": signature,
                "benchmark": contrast_gain(
                    long_contrast,
                    short_contrast,
                    outcome_variance,
                ),
            }
            save_fit(path, cached)

        for row, level in enumerate(list(LEVELS)[1:]):
            benchmark_rows.append(
                {
                    "method": "APOS",
                    "dataset": dataset,
                    "outcome": outcome,
                    "treatment": LEVELS[level],
                    "control_group": group,
                    "ate": float(contrast_summary.loc[row, "coef"]),
                    "se": float(contrast_summary.loc[row, "std err"]),
                    "rv": float(np.ravel(params["rv"])[row]),
                    "rva": float(np.ravel(params["rva"])[row]),
                    "cf_y": float(cached["benchmark"]["cf_y"][row]),
                    "cf_d": float(cached["benchmark"]["cf_d"][row]),
                    "rho": float(cached["benchmark"]["rho"][row]),
                    "delta_theta": float(cached["benchmark"]["delta_theta"][row]),
                }
            )

sens = pd.DataFrame(sens_rows)
benchmarks = pd.DataFrame(benchmark_rows)

sens_table = sens.set_index(
    ["dataset", "outcome", "treatment"]
)[["rv", "rva"]].sort_index()
benchmark_table = benchmarks.set_index(
    ["method", "dataset", "outcome", "treatment", "control_group"]
)[["cf_y", "cf_d", "delta_theta"]].sort_index()

display(Markdown("### Robustness values"))
display(sens_table)
display(Markdown("### Leave-one-control-group-out benchmarks"))
display(benchmark_table)

sens.to_csv(OUT / "results_sensitivity_major_overlap_notebook.csv", index=False)
benchmarks.to_csv(
    OUT / "results_sensitivity_benchmark_major_overlap_notebook.csv",
    index=False,
)
sensitivity_tex(benchmarks)


## Treatment adoption

### Estimation sample


In [ ]:
pred = (
    hh[["water_treatment", *hh_x]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .copy()
)

X = pred[hh_x].to_numpy(dtype=float)
y = pred["water_treatment"].to_numpy(dtype=int)


### Regression models


In [ ]:
logit = sm.Logit(
    y,
    sm.add_constant(X, has_constant="add"),
).fit(disp=0, maxiter=200, method="lbfgs")

lasso = LogisticRegressionCV(
    cv=5,
    penalty="l1",
    solver="saga",
    max_iter=4_000,
    n_jobs=WORKERS,
    random_state=SEED,
    scoring="roc_auc",
).fit(X, y)

enet = LogisticRegressionCV(
    cv=5,
    penalty="elasticnet",
    solver="saga",
    l1_ratios=[0.5],
    max_iter=4_000,
    n_jobs=WORKERS,
    random_state=SEED,
    scoring="roc_auc",
).fit(X, y)


### Tree models


In [ ]:
forest = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=20,
    random_state=SEED,
    n_jobs=WORKERS,
).fit(X, y)

boost = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=SEED,
    n_jobs=WORKERS,
).fit(X, y)


### Predictor table


In [ ]:
predictors = pd.DataFrame(
    {
        "name": hh_x,
        "logit_coef": logit.params[1:],
        "logit_p": logit.pvalues[1:],
        "lasso_coef": lasso.coef_.ravel(),
        "enet_coef": enet.coef_.ravel(),
        "rf_importance": forest.feature_importances_,
        "xgb_importance": boost.feature_importances_,
    }
)
predictor_table = predictors.set_index("name").sort_values(
    ["rf_importance", "xgb_importance"],
    ascending=False,
)

display(Markdown("#### Treatment adoption regressions and variable importance"))
display(predictor_table)

predictors.to_csv(
    OUT / "results_treatment_predictors_major_overlap_notebook.csv",
    index=False,
)
predictor_tex(predictors, pred["water_treatment"])


## Results

### Coefficient plot


In [ ]:
plot = main.dropna(subset=["coef", "se"]).copy()
outcomes = list(plot["outcome"].drop_duplicates())

fig, axes = plt.subplots(
    len(outcomes),
    1,
    figsize=(12, 4 * len(outcomes)),
    squeeze=False,
)

for ax, outcome in zip(axes[:, 0], outcomes):
    part = plot[plot["outcome"] == outcome].copy().reset_index(drop=True)
    labels = part["treatment"].astype(str) + " | " + part["learner"].astype(str)
    y = np.arange(len(part))

    ax.errorbar(
        part["coef"],
        y,
        xerr=1.96 * part["se"],
        fmt="o",
        markersize=3,
        capsize=2,
    )
    ax.axvline(0, color="0.4", linestyle="--", linewidth=0.8)
    ax.set_yticks(y, labels, fontsize=7)
    ax.set_title(outcome)
    ax.set_xlabel("ATE and approximate 95% confidence interval")

fig.tight_layout()
fig.savefig(FIGS / "main_results_major_overlap_notebook.png", dpi=300, bbox_inches="tight")


### Validation checks


In [ ]:
checks = pd.DataFrame(
    [
        {
            "result": "Main",
            "rows": len(main),
            "duplicates": int(main.duplicated().sum()),
            "missing_coef": int(main["coef"].isna().sum()),
        },
        {
            "result": "GATE",
            "rows": len(gate),
            "duplicates": int(gate.duplicated().sum()),
            "missing_coef": int(gate["coef"].isna().sum()),
        },
        {
            "result": "GAPO",
            "rows": len(gapo),
            "duplicates": int(gapo.duplicated().sum()),
            "missing_coef": int(gapo["gapo"].isna().sum()),
        },
        {
            "result": "Sensitivity benchmarks",
            "rows": len(benchmarks),
            "duplicates": int(benchmarks.duplicated().sum()),
            "missing_coef": int(benchmarks["cf_y"].isna().sum()),
        },
    ]
)

display(checks)
